# KernelPrior: LLM-elicited ARD lengthscale priors (Buchwald-Hartwig)

Compares **GP-BO** against the surrogate-level **KernelPrior** architecture
(one-shot LLM elicitation of per-descriptor GP lengthscale weights) and its
iterative variant **KernelPrior-Loop** (re-elicited every 5 iterations using
the GP's own learned lengthscales as feedback), plus **BatchSelect-RAG**
(the ARD-grounded batch selector). 20 seeds x 15 iterations.

Writes `checkpoint_comparacion_kernel_shaping.pkl`, which feeds Figure 7 via
`scripts/generate_figures.py`.

**Discovered inconsistency, preserved deliberately:** this notebook's own
LLM-calling code defaults to `qwen2.5:7b`, even though sibling notebooks'
documentation (and the paper) describe this study as run with `qwen3:30b`.
The `bayesllm.buchwald_hartwig.BuchwaldHartwigBenchmark(model=...)` value
below is set to `qwen2.5:7b` to match the code exactly as it existed in the
original `dia11_prompt_sensitivity.ipynb` -- see
`experiments/README.md#data-lineage-and-reproducibility-notes` for the full
provenance discussion. This was *not* "fixed" here, since doing so would be
a genuine behaviour change, not a refactor.

Renamed from `dia11_prompt_sensitivity.ipynb`. That notebook also contained
a large amount of dead code -- an unused 6-method/10-seed/30-iteration core
grid, unused UCB batch-proposal functions, and an unused single-agent path
-- none of which was ever called by the loop that actually produced the
checkpoint above; none of it is reproduced here. It also defined four
near-duplicate sibling notebooks that split this same run by seed range for
HPC scheduling purposes only (`dia11_batch_llm_grounded_qwen30b*.ipynb`,
`dia11_bo_kernel_shaped*.ipynb`); those are likewise not reproduced here, as
they are redundant with this single notebook.

In [ ]:
from bayesllm.buchwald_hartwig import BuchwaldHartwigBenchmark

bench = BuchwaldHartwigBenchmark(
    data_path="data/Dreher_and_Doyle_input_data.xlsx",
    model="qwen2.5:7b",  # matches the original notebook's code exactly -- see markdown above
)
print(f"Feature space dimensionality: {bench.n_dims}")
print(f"Emulator in-sample R2 (sanity check only): {bench.emulator.score(bench.X_scaled, bench.y_raw):.3f}")

## Main run

In [ ]:
import time
import pickle
import torch

METHODS_COMPARACION_KERNEL = ['bo_puro', 'batch_llm_grounded', 'bo_kernel_shaped', 'bo_kernel_shaped_ard_loop']

CHECKPOINT_PATH_COMPARACION_KERNEL = "checkpoint_comparacion_kernel_shaping.pkl"
SEEDS_COMPARACION_KERNEL = list(range(20))
N_ITER_COMPARACION_KERNEL = 15

all_results_comparacion_kernel = []

for seed in SEEDS_COMPARACION_KERNEL:
    X_init, Y_init, history_init = bench.generate_initial_design(seed)
    for metodo in METHODS_COMPARACION_KERNEL:
        print(f"=== Seed {seed} | metodo: {metodo} ===")
        t0 = time.time()
        X_bo, Y_bo, historial = bench.clone_run_state(X_init, Y_init, history_init)
        weights_state = {'pesos': None, 'reasoning': '', 'ultima_elicitacion': -bench.KERNEL_ARD_REELICIT_EVERY}

        for it in range(N_ITER_COMPARACION_KERNEL):
            if metodo == 'bo_puro':
                resultado = bench.run_bo_only_iteration(X_bo, Y_bo, seed=seed * 1000 + it)
            elif metodo == 'batch_llm_grounded':
                resultado = bench.run_batch_variant_iteration(
                    X_bo, Y_bo, historial, prompt_variant='grounded', iteracion=it
                )
            elif metodo in ('bo_kernel_shaped', 'bo_kernel_shaped_ard_loop'):
                resultado = bench.run_kernel_shaping_iteration(metodo, X_bo, Y_bo, it, seed, weights_state)
            else:
                raise ValueError(f"Metodo desconocido: {metodo}")

            X_bo = torch.cat([X_bo, resultado['x']])
            Y_bo = torch.cat([Y_bo, resultado['y']])
            historial.append({'descriptors': bench.tensor_to_dict(resultado['x']), 'yield': resultado['y'].item()})

            best_so_far = Y_bo.max().item()
            all_results_comparacion_kernel.append({
                'seed': seed, 'metodo': metodo, 'iteracion': it,
                'yield': resultado['y'].item(), 'best_so_far': best_so_far,
                'fuente': resultado.get('fuente'), 'rechazos': resultado.get('rechazos', 0),
            })

        elapsed = (time.time() - t0) / 60
        print(f"  listo en {elapsed:.1f} min, best_so_far final: {best_so_far:.2f}%")

        with open(CHECKPOINT_PATH_COMPARACION_KERNEL, "wb") as f:
            pickle.dump(all_results_comparacion_kernel, f)

print("Comparacion terminada.")

## Post-hoc analysis: win/loss counts and paired Wilcoxon vs. GP-BO

In [ ]:
import pandas as pd
from scipy.stats import wilcoxon

df_comp = pd.DataFrame(all_results_comparacion_kernel)
final_iter = df_comp['iteracion'].max()
final_comp = df_comp[df_comp['iteracion'] == final_iter][['seed', 'metodo', 'best_so_far']]

resumen = final_comp.groupby('metodo')['best_so_far'].agg(['mean', 'std']).round(2)
print(resumen)

pivot_comp = final_comp.pivot(index='seed', columns='metodo', values='best_so_far')
for metodo in ['batch_llm_grounded', 'bo_kernel_shaped', 'bo_kernel_shaped_ard_loop']:
    diffs = (pivot_comp[metodo] - pivot_comp['bo_puro']).dropna()
    stat, p = wilcoxon(diffs)
    victorias = int((diffs > 0).sum())
    derrotas = int((diffs < 0).sum())
    print(f"{metodo} vs bo_puro: n={len(diffs)}, media_diff={diffs.mean():.2f}, "
          f"victorias={victorias}, derrotas={derrotas}, p={p:.4f}")